# Validate the CISSP practice bank (Colab)

This notebook:
1. Runs the same checks as `validation/validate.py` (schema, IDs, counts, answer letters, length bias, near-duplicates).
2. Optionally runs an **independent answer-key audit**: a Claude model answers each question without seeing the key, explanation, or rationales. Disagreements, low-confidence answers, and items where another option looks defensible go to `validation/flags.csv` for human review.

**API key:** add `ANTHROPIC_API_KEY` in Colab Secrets (key icon in the left sidebar) and turn on notebook access. The key is read with `userdata.get` and is never printed or written to disk.

**Cost:** `MODE = "batch"` uses the Message Batches API at half price and usually finishes within an hour. `MODE = "sync"` answers one question at a time and can resume after an interruption. Try `LIMIT = 5` first as a smoke test.

In [ ]:
REPO_URL = "https://github.com/RickPack/cissp-practice-diagnostic.git"
BRANCH = "main"

MODEL = "claude-sonnet-5"   # change this to audit with a different Claude model
EFFORT = "high"             # low | medium | high | max
TARGET = "all"              # "all" for the full bank, or one batch file such as "data/batches/D1_2.json"
MODE = "batch"              # "batch" (half price, asynchronous) or "sync" (immediate, resumable)
CONFIDENCE_THRESHOLD = 70   # answers below this confidence are flagged
LIMIT = 0                   # >0 audits only the first N questions

In [ ]:
import os, sys
if not os.path.isdir("cissp-practice-diagnostic"):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL}
%cd cissp-practice-diagnostic
!pip install -q -r validation/requirements.txt -r validation/requirements-audit.txt
sys.path.insert(0, "validation")

## Step 1: Structural validation

If `TARGET` is a batch file that is not in the manifest yet, it is validated as a staged batch alongside the full bank.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
import validate

include = [] if TARGET == "all" else [validate.ROOT / TARGET]
result, taxonomy = validate.run_checks(validate.ROOT, include)
display(Markdown(validate.render_report(result, taxonomy)))
print("PASS" if result.ok else "FAIL: fix the hard failures before auditing")

## Step 2: Independent answer-key audit (optional)

In [ ]:
from google.colab import userdata
import anthropic

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

In [ ]:
import audit

questions = audit.load_questions(audit.ROOT, TARGET)
if LIMIT:
    questions = questions[:LIMIT]
print(f"Auditing {len(questions)} questions with {MODEL} in {MODE} mode")

if MODE == "batch":
    results = audit.audit_batch(client, questions, MODEL, EFFORT)
else:
    results = audit.audit_sync(client, questions, MODEL, EFFORT, cache=audit.ROOT / "validation" / ".audit_cache.json")

flags = audit.make_flags(questions, results, MODEL, CONFIDENCE_THRESHOLD)
audit.write_flags(flags, audit.ROOT / "validation" / "flags.csv", {q["id"] for q in questions})
print(audit.summarize(questions, results, flags))

In [ ]:
import pandas as pd

flag_table = pd.read_csv("validation/flags.csv")
display(flag_table[["id", "key", "model_answer", "confidence", "other_defensible", "reasons", "item_issue"]])

In [ ]:
from google.colab import files

files.download("validation/flags.csv")

## Next steps

1. Copy the downloaded `flags.csv` into `validation/` in your local clone.
2. For each row, correct or replace the question, then write what you did in the `resolution` column.
3. Re-run `python validation/validate.py`, commit, and push. CI redeploys the site.